<a href="https://colab.research.google.com/github/Skylar-siyu/BigData/blob/main/%E2%80%9CW05_quiz_2026_ipynb%E2%80%9D%E7%9A%84%E5%89%AF%E6%9C%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 5 Quiz

This notebook contains the SQL Quiz for Week 5. Section 1 uses the New York City data we know and love. Section 2 uses building footprint data from the Google Open Buildings dataset.

INSTRUCTIONS:

Run this notebook in Google Colab. The answer to each quesiton will be a number or a string. Input these into the corresponding question on Moodle. You have 90 minutes to attempt the quiz, so if you get stuck on a question, move on.

Make sure you run all of the cells of code in order, especially the ones that already have code in them! If you run into serious problems, try clicking on the "runtime" tab above and selecting "restart session and run all".
Use the cells below for any necessary setup.

# Section 1



The URL below points to a zip file containing the data on New York City that we've been working with so far.

https://s3.amazonaws.com/s3.cleverelephant.ca/postgis-workshop-2020.zip

## Question 1

Create the following tables using the corresponding shapefiles.
- nyc_neighborhoods
- nyc_census_blocks
- nyc_homicides
- nyc_streets
- nyc_subway_stations

In [1]:
%pip install duckdb leafmap duckdb-engine jupysql lonboard
import duckdb
import leafmap
import pandas as pd
import os
import zipfile
import requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 667.8/667.8 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.4/20.4 MB 69.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 89.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.8/192.8 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 85.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.0 MB/

In [2]:
url = "https://s3.amazonaws.com/s3.cleverelephant.ca/postgis-workshop-2020.zip"
zip_path = "nyc_data.zip"
r = requests.get(url)
with open(zip_path, "wb") as f:
    f.write(r.content)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("nyc_data")

os.listdir("nyc_data")

['postgis-workshop']

In [3]:
os.listdir("nyc_data/postgis-workshop/data")

['nyc_homicides.prj',
 'nyc_streets.dbf',
 'nyc_homicides.shp',
 'nyc_streets.shp',
 'nyc_census_blocks.shx',
 'nyc_neighborhoods.shx',
 'nyc_census_blocks.dbf',
 'nyc_streets.shx',
 'nyc_subway_stations.shp',
 'nyc_neighborhoods.prj',
 'nyc_homicides.shx',
 'nyc_data.backup',
 'nyc_streets.prj',
 'nyc_subway_stations.shx',
 'nyc_census_blocks.shp',
 'nyc_subway_stations.dbf',
 'nyc_neighborhoods.shp',
 'nyc_subway_stations.prj',
 'nyc_homicides.dbf',
 'nyc_neighborhoods.dbf',
 'nyc_census_sociodata.sql',
 '2000',
 'nyc_census_blocks.prj']

In [4]:
con = duckdb.connect()
con.install_extension("httpfs")
con.load_extension("httpfs")
con.install_extension("spatial")
con.load_extension("spatial")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [5]:
tables = [
    "nyc_neighborhoods",
    "nyc_census_blocks",
    "nyc_homicides",
    "nyc_streets",
    "nyc_subway_stations"
]

for table in tables:
    shp_file = f"nyc_data/postgis-workshop/data/{table}.shp"
    con.execute(f"""
        CREATE TABLE {table} AS
        SELECT * FROM ST_Read('{shp_file}');
    """)

In [6]:
print(con.execute("SHOW TABLES;").fetchall())

[('nyc_census_blocks',), ('nyc_homicides',), ('nyc_neighborhoods',), ('nyc_streets',), ('nyc_subway_stations',)]


## Question 2:
How many subway stations are there in NYC?



In [7]:
con.sql("""
    SELECT COUNT(*)
    FROM nyc_subway_stations;
""")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│          491 │
└──────────────┘

## Question 3:

What combination of borough and year hard the largest number of homicide victims?

In [8]:
con.sql("""
    SELECT *
    FROM nyc_homicides
    LIMIT 5;
""").df()


,INCIDENT_D,BORONAME,NUM_VICTIM,PRIMARY_MO,ID,WEAPON,LIGHT_DARK,YEAR,geom
0,2008-01-01,Brooklyn,1,None,7,gun,D,2008,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
1,2008-01-04,Manhattan,1,None,14,gun,D,2008,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
2,2008-01-05,Queens,1,None,15,gun,D,2008,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
3,2008-01-04,Queens,1,None,16,knife,D,2008,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
4,2008-01-05,Queens,1,None,18,gun,D,2008,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."


In [9]:
con.sql("""
    SELECT
        BORONAME,
        YEAR,
        SUM(CAST(NUM_VICTIM AS INTEGER)) AS total_victims
    FROM nyc_homicides
    GROUP BY BORONAME, YEAR
    ORDER BY total_victims DESC
    LIMIT 1;
""")

┌──────────┬───────┬───────────────┐
│ BORONAME │ YEAR  │ total_victims │
│ varchar  │ int64 │    int128     │
├──────────┼───────┼───────────────┤
│ Brooklyn │  2003 │           235 │
└──────────┴───────┴───────────────┘

## Question 4:

Which subway station has the largest number of shooting victims within a 300 meter radius?


In [10]:
con.sql("""
    SELECT
        s.name AS station_name,
        SUM(CAST(h.NUM_VICTIM AS INTEGER)) AS total_victims
    FROM nyc_subway_stations s
    JOIN nyc_homicides h
      ON ST_DWithin(
          s.geom,
          h.geom,
          300
      )
    WHERE h.WEAPON = 'gun'
    GROUP BY s.name
    ORDER BY total_victims DESC
    LIMIT 1;
""")

┌──────────────┬───────────────┐
│ station_name │ total_victims │
│   varchar    │    int128     │
├──────────────┼───────────────┤
│ Franklin Ave │            28 │
└──────────────┴───────────────┘

## Question 5

what is the most densely populated residential street in new york? (total population of intersecting census blocks divided by the total length of the street)

In [15]:
con.sql("""
    SELECT
        s.name AS street_name,
        SUM(CAST(c.popn_total AS INTEGER)) / ST_Length(s.geom) AS pop_density
    FROM nyc_streets s
    JOIN nyc_census_blocks c
      ON ST_Intersects(s.geom, c.geom)
    WHERE s.type = 'residential'
    GROUP BY s.name, s.geom
    HAVING ST_Length(ST_Union_Agg(s.geom)) > 0
    ORDER BY pop_density DESC
    LIMIT 1;
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,street_name,pop_density
0,W 66 St,3223.78748


# Section 2

This section uses internet speed data from [Ookla](https://github.com/teamookla/ookla-open-data); take a minute to read the documentation of the dataset.

Create a table called ookla_nyc which reads the data from this url:

https://storage.googleapis.com/qm2/CASA0025/ookla_nyc.parquet




In [13]:
con.sql("""
    CREATE TABLE ookla_nyc AS
    SELECT *
    FROM 'https://storage.googleapis.com/qm2/CASA0025/ookla_nyc.parquet';
""")

In [14]:
con.sql("SHOW TABLES;")

┌─────────────────────┐
│        name         │
│       varchar       │
├─────────────────────┤
│ nyc_census_blocks   │
│ nyc_homicides       │
│ nyc_neighborhoods   │
│ nyc_streets         │
│ nyc_subway_stations │
│ ookla_nyc           │
└─────────────────────┘

In [18]:
con.sql("SELECT * FROM ookla_nyc LIMIT 5;").df()

,quadkey,tile_x,tile_y,avg_d_kbps,avg_u_kbps,avg_lat_ms,avg_lat_down_ms,avg_lat_up_ms,tests,devices,quarter,type,year,geometry
0,0320101100103011,-74.2923,40.9446,725128,135953,11,57,216,6,3,3,fixed,2025,"POLYGON ((559335.820448 4533081.030772, 559798..."
1,0320101100103100,-74.2868,40.9446,119511,14121,12,814,210,7,1,3,fixed,2025,"POLYGON ((559798.18087 4533084.773678, 560260...."
2,0320101100103101,-74.2813,40.9446,239400,33087,19,123,86,2,2,3,fixed,2025,"POLYGON ((560260.541371 4533088.54564, 560722...."
3,0320101100103103,-74.2813,40.9405,351406,314161,3,12,26,1,1,3,fixed,2025,"POLYGON ((560264.313317 4532627.94173, 560726...."
4,0320101100103110,-74.2758,40.9446,327591,153403,6,315,155,5,3,3,fixed,2025,"POLYGON ((560722.90195 4533092.346658, 561185...."


## Question 6
How many tiles are contained within new york city neighbourhoods?

In [21]:
con.sql("""
    SELECT COUNT(DISTINCT o.quadkey) AS num_tiles
    FROM ookla_nyc o
    JOIN nyc_neighborhoods n
      ON ST_Within(
           ST_GeomFromText(o.geometry),
           n.geom
         );
""")


┌───────────┐
│ num_tiles │
│   int64   │
├───────────┤
│      1403 │
└───────────┘

## Question 7

Which neighbourhood has the highest average download speed? use 'within' rather than intersects, and ignore tiles with fewer than 3 tests.

In [22]:
con.sql("""
    SELECT
        n.name AS neighbourhood,
        AVG(o.avg_d_kbps) / 1000.0 AS avg_download_mbps
    FROM ookla_nyc o
    JOIN nyc_neighborhoods n
      ON ST_Within(
           ST_GeomFromText(o.geometry),
           n.geom
         )
    WHERE o.tests >= 3
    GROUP BY n.name
    ORDER BY avg_download_mbps DESC
    LIMIT 1;
""")

┌───────────────┬───────────────────┐
│ neighbourhood │ avg_download_mbps │
│    varchar    │      double       │
├───────────────┼───────────────────┤
│ Woodside      │ 455.9603333333333 │
└───────────────┴───────────────────┘

## Question 8



What is the name of the subway station closest to the tile with the highest download speed?

In [23]:
con.sql("""
    WITH fastest_tile AS (
        SELECT
            ST_GeomFromText(geometry) AS geom
        FROM ookla_nyc
        ORDER BY avg_d_kbps DESC
        LIMIT 1
    )
    SELECT
        s.name AS station_name
    FROM nyc_subway_stations s, fastest_tile f
    ORDER BY ST_Distance(s.geom, f.geom) ASC
    LIMIT 1;
""")


┌──────────────┐
│ station_name │
│   varchar    │
├──────────────┤
│ 190th St     │
└──────────────┘

## Question 9

Which neighbourhood has the highest number of devices per capita? Use only spatial intersections.

In [24]:
con.sql("""
    SELECT
        n.name AS neighbourhood,
        SUM(o.devices) * 1.0 / SUM(c.popn_total) AS devices_per_capita
    FROM nyc_neighborhoods n
    JOIN ookla_nyc o
      ON ST_Intersects(
          ST_GeomFromText(o.geometry),
          n.geom
      )
    JOIN nyc_census_blocks c
      ON ST_Intersects(c.geom, n.geom)
    GROUP BY n.name
    ORDER BY devices_per_capita DESC
    LIMIT 1;
""")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────────┬────────────────────┐
│ neighbourhood │ devices_per_capita │
│    varchar    │       double       │
├───────────────┼────────────────────┤
│ Flatbush      │                inf │
└───────────────┴────────────────────┘

## Question 10


What is the average upload speed for tiles that are within 200 meters of L train stops and intersect with census blocks where the black population is above 50%?

In [26]:
con.sql("""
    SELECT
        AVG(o.avg_u_kbps) AS avg_upload_speed_kbps
    FROM ookla_nyc o
    JOIN nyc_subway_stations s
      ON ST_DWithin(
          ST_GeomFromText(o.geometry),
          s.geom,
          200
      )
    JOIN nyc_census_blocks c
      ON ST_Intersects(
          ST_GeomFromText(o.geometry),
          c.geom
      )
    WHERE s.ROUTES LIKE '%L%'
      AND c.popn_black * 1.0 / c.popn_total > 0.5;
""")


┌───────────────────────┐
│ avg_upload_speed_kbps │
│        double         │
├───────────────────────┤
│    124413.30695443646 │
└───────────────────────┘